# 🧩 AI Agents + Skills — Wissen on demand (Demo: TuttiPaletti)

Bisher kamen die *Fähigkeiten* des Agenten von **Tools** (lokal oder via MCP). Jetzt kommt das *Vorgehen* aus **Skills**: vorgefertigte Arbeitsanweisungen, die der Agent nur dann lädt, wenn er sie braucht.

Ein **Skill** ist einfach ein Ordner mit einer `SKILL.md`:
- **Frontmatter** (`name`, `description`) — kurz, beschreibt **wann** der Skill greift. Nur das ist permanent im Kontext.
- **Body** — die ausführliche Schritt-für-Schritt-Anleitung. Wird **on demand** geladen (*progressive disclosure*).

Dazu eine **`CLAUDE.md`**: der immer geladene Einstieg mit den verbindlichen Regeln.

| Alles im System-Prompt | Skills (on demand) |
|---|---|
| jede Anleitung dauerhaft im Kontext | nur kurze Beschreibungen dauerhaft |
| Prompt wächst mit jeder Fähigkeit | bleibt klein; Details bei Bedarf |
| Token-teuer | Token-effizient |
| schwer zu pflegen | je Skill **eine Datei** |

```
  Aufgabe → Agent
    1) CLAUDE.md            (immer geladen: Rolle + Regeln)
    2) list_skills()        → [name + description]      (klein, immer)
    3) passt? read_skill()  → ganze SKILL.md            (groß, nur bei Bedarf)
    4) folgt der Anleitung, nutzt Datei-Tools (read_file / write_file)
```

Als echte Demo nehmen wir **[TuttiPaletti](.)** (BTK-Palettenklärung): ein Sachbearbeiter-Setup, das Kundenrückfragen zu Palettenrechnungen vorbereitet — **immer nur als Draft**, nie gesendet. Unser *from-scratch*-Agent nutzt **dieselben** Skills wie Claude Code / Cowork, hier lokal über `test-mails/` statt Gmail.

> Es bleibt **derselbe Agent-Loop** wie in den anderen Notebooks. Neu ist nur: Der Agent **entdeckt** und **lädt** Skills, statt alles fest im Prompt zu haben.

## 0 · Setup

`llm()` wie gewohnt (Azure OpenAI aus der `.env`). Zusätzlich zeigen wir mit `ROOT` auf das **TuttiPaletti-Repo** — dort liegen `CLAUDE.md`, `skills/`, `kontext/`, `test-mails/`.

In [1]:
import os, sys, json, re
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

def llm(messages, tools=None, tool_choice="auto"):
    """Unser einziger Draht zum Modell - identisch zu den anderen Notebooks."""
    kwargs = dict(model=DEPLOYMENT, messages=messages)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

# Projektwurzel = das TuttiPaletti-Repo (bei Bedarf anpassen).
ROOT = Path(r"C:\Users\rudi\source\repos\TuttiPaletti")
assert (ROOT / "CLAUDE.md").exists(), f"CLAUDE.md nicht gefunden unter {ROOT}"
print("Setup bereit. Deployment:", DEPLOYMENT)
print("ROOT:", ROOT)

Setup bereit. Deployment: gpt-5.4-mini
ROOT: C:\Users\rudi\source\repos\TuttiPaletti


## 1 · Skill-Discovery — nur die Beschreibungen laden

Der Kern von *progressive disclosure*: Wir parsen aus jeder `SKILL.md` **nur das Frontmatter** (`name` + `description`). Das ist der schlanke Index, der permanent im Kontext liegen kann. Die vollständige Anleitung holt der Agent erst, wenn ein Skill wirklich passt.

In [2]:
SKILLS_DIR = ROOT / "skills"

def _frontmatter(text):
    """Liest den YAML-Block zwischen den ersten beiden '---' (einzeilige Werte)."""
    if not text.startswith("---"):
        return {}
    end = text.find("\n---", 3)
    meta = {}
    for line in text[3:end].splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    return meta

def skill_index():
    out = []
    for p in sorted(SKILLS_DIR.glob("*/SKILL.md")):
        fm = _frontmatter(p.read_text(encoding="utf-8"))
        out.append({"name": fm.get("name", p.parent.name), "description": fm.get("description", "")})
    return out

idx = skill_index()
print("Verfuegbare Skills (nur Frontmatter -> kann immer im Kontext liegen):\n")
for s in idx:
    print(f"• {s['name']}: {s['description'][:120]}...")

full  = sum(len(p.read_text(encoding="utf-8")) for p in SKILLS_DIR.glob("*/SKILL.md"))
index = len(json.dumps(idx, ensure_ascii=False))
print(f"\nProgressive Disclosure: Index {index} Zeichen  vs  alle SKILL.md zusammen {full} Zeichen")
print(f"-> nur ~{index*100//full}% ist permanent im Kontext, der Rest wird bei Bedarf geladen.")

Verfuegbare Skills (nur Frontmatter -> kann immer im Kontext liegen):

• dummy-mapping: Belege aus `kontext/dummy-belege/`, die im Dummy-Palettenkonto haengen, dem richtigen Kunden zuordnen. Liest jeden Beleg...
• rechnungsrueckfrage: Vorbearbeitung von Kundenrueckfragen zu Palettenrechnungen, Saldo und Belegen. Klassifiziert eingehende Gmail-Mails im L...

Progressive Disclosure: Index 1057 Zeichen  vs  alle SKILL.md zusammen 10371 Zeichen
-> nur ~10% ist permanent im Kontext, der Rest wird bei Bedarf geladen.


## 2 · Die Werkzeuge des Agenten

Fünf Tools — zwei für **Skills** (Discovery + Laden), drei für **Dateien** (erkunden, lesen, schreiben). Alle Pfade sind auf `ROOT` eingesperrt (kein Ausbruch aus dem Projekt).

Der Systemprompt = die **immer geladene** `CLAUDE.md` + eine kleine **Demo-Notiz**, die den Gmail-Workflow auf lokale Dateien umbiegt (`test-mails/` statt Gmail, `drafts/` statt Gmail-Draft). Den eigentlichen Skill ändern wir **nicht** — nur die Ein-/Ausgabe.

In [3]:
def _safe(path):
    p = (ROOT / path).resolve()
    if p != ROOT.resolve() and ROOT.resolve() not in p.parents:
        raise ValueError(f"Pfad ausserhalb des Projekts: {path}")
    return p

def list_dir(path="."):
    p = _safe(path)
    if not p.exists(): return f"(nicht gefunden: {path})"
    return "\n".join(sorted(("[dir] " if c.is_dir() else "      ") + c.name for c in p.iterdir()))

def read_file(path):
    p = _safe(path)
    return p.read_text(encoding="utf-8") if p.exists() else f"(Datei nicht gefunden: {path})"

def write_file(path, content):
    p = _safe(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return f"OK: {path} ({len(content)} Zeichen) geschrieben"

def read_skill(name):
    p = SKILLS_DIR / name / "SKILL.md"
    return p.read_text(encoding="utf-8") if p.exists() else f"(kein Skill '{name}')"

def list_skills():
    return json.dumps(skill_index(), ensure_ascii=False, indent=2)

TOOLS = [
  {"type": "function", "function": {"name": "list_skills",
    "description": "Listet verfuegbare Skills (Name + Beschreibung). ZUERST aufrufen, um den passenden Skill zu finden.",
    "parameters": {"type": "object", "properties": {}}}},
  {"type": "function", "function": {"name": "read_skill",
    "description": "Laedt die vollstaendige Anleitung (SKILL.md) eines Skills.",
    "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}}},
  {"type": "function", "function": {"name": "list_dir",
    "description": "Listet den Inhalt eines Ordners (relativ zur Projektwurzel).",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}},
  {"type": "function", "function": {"name": "read_file",
    "description": "Liest eine Datei (relativ zur Projektwurzel).",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}},
  {"type": "function", "function": {"name": "write_file",
    "description": "Schreibt eine Datei (relativ zur Projektwurzel), z. B. fall-log/... oder drafts/...",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]}}},
]

DISPATCH = {
  "list_skills": lambda a: list_skills(),
  "read_skill":  lambda a: read_skill(a["name"]),
  "list_dir":    lambda a: list_dir(a.get("path", ".")),
  "read_file":   lambda a: read_file(a["path"]),
  "write_file":  lambda a: write_file(a["path"], a["content"]),
}
def dispatch(name, args):
    try:    return DISPATCH[name](args)
    except Exception as e: return f"Fehler in {name}: {e}"

CLAUDE_MD = (ROOT / "CLAUDE.md").read_text(encoding="utf-8")

SYSTEM = CLAUDE_MD + """

---
## Demo-Umgebung (lokal statt Cowork/Gmail)
Du arbeitest hier ueber lokale Dateien:
- Eingehende Mails liegen als `.eml` in `test-mails/` (es gibt KEIN Gmail).
- Statt eines Gmail-Drafts schreibst du den Antwort-Entwurf nach `drafts/<YYYY-MM-DD>-<kunde-slug>.md` (erste Zeile = `Betreff: ...`).
- Statt Gmail-Labels notierst du den Status nur im Abschnitt "Draft-Status" der Falldatei.

## Arbeitsweise (Skills)
1. Rufe `list_skills` auf und waehle den passenden Skill.
2. Lade ihn mit `read_skill(name)` und folge seiner Anleitung EXAKT (Falldatei-Format, Evidenzpflicht, Eskalationsregeln, Stil aus `beispiel-antworten/`).
3. Sammle Evidenz mit `list_dir`/`read_file` aus `kontext/`.
4. Schreibe Falldatei (`fall-log/...`) und Draft (`drafts/...`) mit `write_file`. Erfinde nichts; fehlende Belege offen benennen und gemaess Skill eskalieren.
Am Ende: kurze Zusammenfassung im Chat.
"""
print("Tools + Systemprompt bereit. CLAUDE.md:", len(CLAUDE_MD), "Zeichen.")

Tools + Systemprompt bereit. CLAUDE.md: 5374 Zeichen.


## 3 · Der Agent-Loop

Exakt die Schleife aus den anderen Notebooks: fragen → Tool ausführen → Ergebnis anhängen → erneut fragen. Wir drucken jeden Tool-Aufruf mit, damit man **Discovery → Laden → Ausführen** live sieht.

In [4]:
def run_skill_agent(task, max_steps=30, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        msg = llm(messages, tools=TOOLS).choices[0].message
        a = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            a["tool_calls"] = [{"id": tc.id, "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
        messages.append(a)
        if not msg.tool_calls:
            if verbose: print(f"[{step}] ✓ fertig")
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            short = {k: (v[:45] + "…" if isinstance(v, str) and len(v) > 45 else v) for k, v in args.items()}
            if verbose: print(f"[{step}] → {tc.function.name}({short})")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": dispatch(tc.function.name, args)})
    return "(max_steps erreicht)"

def show_newest(folder, n=1):
    files = sorted((ROOT / folder).glob("*.md"), key=lambda p: p.stat().st_mtime, reverse=True)[:n]
    for f in files:
        print(f"\n===== {f.relative_to(ROOT)} =====\n")
        print(f.read_text(encoding="utf-8"))

## 4 · Demo — eine Rechnungsrückfrage verarbeiten

Wir geben dem Agenten **eine** Test-Mail. Beobachte die Spur: `list_skills` → `read_skill(rechnungsrueckfrage)` → er liest `kundenstamm.csv`, das passende Palettenkonto, die Rechnung und Belege in `kontext/` → schreibt **Falldatei** und **Draft**.

In [5]:
antwort = run_skill_agent("Verarbeite die Mail test-mails/01-mueller-bau-rechnungsfrage.eml")
print("\n=== Zusammenfassung des Agenten ===\n", antwort)

# Die erzeugten Artefakte:
show_newest("fall-log")
show_newest("drafts")

[1] → list_skills({})
[2] → read_skill({'name': 'rechnungsrueckfrage'})
[3] → list_dir({'path': 'test-mails'})
[3] → list_dir({'path': 'kontext/kundenstamm.csv'})
[3] → list_dir({'path': 'kontext/palettenkonten'})
[3] → list_dir({'path': 'kontext/rechnungen'})
[3] → list_dir({'path': 'kontext/belege'})
[3] → list_dir({'path': 'beispiel-antworten'})
[4] → read_file({'path': 'test-mails/01-mueller-bau-rechnungsfrage.eml'})
[4] → read_file({'path': 'kontext/kundenstamm.csv'})
[4] → read_file({'path': 'kontext/palettenkonten/K10847_mueller-bau.csv'})
[4] → read_file({'path': 'kontext/rechnungen/RG-2026-04-0299.md'})
[4] → read_file({'path': 'kontext/belege/ABL-99381.md'})
[4] → read_file({'path': 'beispiel-antworten/01-rechnungsfrage.md'})
[5] → read_file({'path': 'kontext/rechnungen/RG-2026-04-0312.md'})
[6] → write_file({'path': 'fall-log/2026-05-14-mueller-bau.md', 'content': '# Fall 2026-05-14-mueller-bau\n\n## Mail\n- **Vo…'})
[7] → write_file({'path': 'drafts/2026-05-14-mueller-bau-b

## 5 · Demo — Eskalation (mehrdeutiger Kunde)

Jetzt eine knifflige Mail: **Hofer Logistik** (K12001) vs. **Hofer Lagerhaus** (K12044) — im Kundenstamm ausdrücklich als verwechslungsgefährdet markiert. Hier soll der Agent **nicht raten**, sondern gemäß den Regeln in `CLAUDE.md`/Skill **eskalieren** (Draft mit `BITTE PRUEFEN: ...`).

In [6]:
antwort = run_skill_agent("Verarbeite die Mail test-mails/05-hofer-mehrdeutig.eml")
print("\n=== Zusammenfassung des Agenten ===\n", antwort)

show_newest("fall-log")
show_newest("drafts")

[1] → list_skills({})
[2] → read_skill({'name': 'rechnungsrueckfrage'})
[3] → read_file({'path': 'test-mails/05-hofer-mehrdeutig.eml'})
[3] → list_dir({'path': 'kontext'})
[3] → list_dir({'path': 'beispiel-antworten'})
[3] → list_dir({'path': 'fall-log'})
[4] → read_file({'path': 'beispiel-antworten/02-saldoanfrage.md'})
[4] → read_file({'path': 'kontext/kundenstamm.csv'})
[5] → write_file({'path': 'fall-log/2026-05-15-hofer-mehrdeutig.md', 'content': '# Fall 2026-05-15-hofer-mehrdeutig\n\n## Mail\n-…'})
[6] → write_file({'path': 'drafts/2026-05-15-hofer-mehrdeutig.md', 'content': 'Betreff: Bitte um Konkretisierung zur Zuordnu…'})
[7] ✓ fertig

=== Zusammenfassung des Agenten ===
 1 Mail verarbeitet:
- 0 Entwurf-bereit
- 1 eskaliert
- 0 Sonstiges

Grund: Absender nicht eindeutig zuordenbar, zwei plausible Hofer-Kunden im Stammsatz.

Falldatei und Draft wurden angelegt.

===== fall-log\2026-05-15-hofer-mehrdeutig.md =====

# Fall 2026-05-15-hofer-mehrdeutig

## Mail
- **Von**: Andreas H

## 6 · Interaktiv

Gib dem Agenten selbst eine Aufgabe. Beispiele:
- `Verarbeite die Mail test-mails/03-steiner-saldoanfrage.eml`
- `Verarbeite alle Mails in test-mails/`
- `Welche Skills hast du und wofuer?`

Leere Eingabe oder `exit` beendet.

In [ ]:
while True:
    frage = input("\U0001f9d1 Aufgabe (oder 'exit'): ").strip()
    if not frage or frage.lower() in ("exit", "quit", "ende"):
        print("Beendet."); break
    antwort = run_skill_agent(frage)
    print("\n\U0001f916", antwort, "\n")

## Mitnehmen

1. **Ein Skill ist nur eine Datei.** Ordner + `SKILL.md` (Frontmatter `name`/`description` + Anleitung). Kein Framework, kein Magie.
2. **Progressive Disclosure.** Permanent im Kontext liegen nur die **Beschreibungen**; die ausführliche Anleitung wird per `read_skill` **bei Bedarf** geladen. So skaliert es auf beliebig viele Skills, ohne den Prompt zu sprengen.
3. **`CLAUDE.md` = Einstieg + Regeln**, immer geladen. **Skills = abrufbares Detailwissen**. Genau dieses Muster nutzen Claude Code / Cowork — wir haben es hier *from scratch* nachgebaut.
4. **Derselbe Loop.** „Skill" ist kein neuer Agenten-Mechanismus, sondern **Wissen/Vorgehen als Datei**, das der Agent über Tools entdeckt, lädt und befolgt.
5. **Tools vs. Skills.** Tools/MCP = **Fähigkeiten** (was der Agent *tun* kann). Skills = **Wissen/Vorgehen** (*wie* er es tut). Zusammen ergeben sie einen verlässlichen Sachbearbeiter — inklusive der im Skill kodierten Leitplanken (nur Drafts, Evidenzpflicht, Eskalation).

> **Sicherheit by design:** Die Regeln „nie senden, nie buchen, nichts erfinden, bei Unklarheit eskalieren" stehen im Skill/`CLAUDE.md` — der Agent **befolgt** sie, statt sie raten zu müssen.